# Course–Job Embedding Pipeline

Runs the embedding pipeline on Colab's GPU. All logic lives in `src/embedding/embed.py` — this notebook is just a launcher.

### Setup checklist
1. **GPU**: Runtime → Change runtime type → **T4 GPU**
2. **Data**: The processed CSVs need to be accessible from Google Drive (see below).

### Getting OneDrive data into Google Drive
Colab can’t mount OneDrive natively. The easiest workaround:
1. Open the shared OneDrive folder (`DSA4264_Project_Data/`)
2. Download `processed/jobs/03_jobs_filtered.csv` and `processed/courses/modules_cleaned.csv`
3. Upload them to Google Drive, preserving the folder structure:
   ```
   MyDrive/DSA4264_Project_Data/
     processed/jobs/03_jobs_filtered.csv
     processed/courses/modules_cleaned.csv
     embeddings/    ← outputs will be saved here
   ```
4. After the pipeline runs, download the `embeddings/` folder from Drive and copy it back to OneDrive for teammates.

This is a one-time setup. If the processed data changes, just re-upload the two CSVs.

In [1]:
# 1. Mount Google Drive & set data path
from google.colab import drive
drive.mount('/content/drive')

# Point this to wherever you put the shared data folder in Drive
DATA_ROOT = "/content/drive/MyDrive/DSA4264_Project_Data"  # <-- CHANGE IF NEEDED

Mounted at /content/drive


In [2]:
# 2. Verify data files exist
import os

required = [
    f"{DATA_ROOT}/processed/jobs/03_jobs_filtered.csv",
    f"{DATA_ROOT}/processed/courses/modules_cleaned.csv",
]

all_good = True
for f in required:
    exists = os.path.exists(f)
    print(f"  {'✓' if exists else '✗ MISSING'}  {f}")
    if not exists:
        all_good = False

if not all_good:
    raise FileNotFoundError(
        "Missing data files. See instructions above for uploading "
        "processed CSVs from OneDrive to Google Drive."
    )

# Create embeddings output directory
os.makedirs(f"{DATA_ROOT}/embeddings", exist_ok=True)
print("\n  Data verified. Ready to run.")

  ✓  /content/drive/MyDrive/DSA4264_Project_Data/processed/jobs/03_jobs_filtered.csv
  ✓  /content/drive/MyDrive/DSA4264_Project_Data/processed/courses/modules_cleaned.csv

  Data verified. Ready to run.


In [3]:
# 3. Clone repo & install dependencies
REPO_URL = "https://github.com/DanDmc/DSA4264_Project.git"  # <-- CHANGE IF NEEDED
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

!git clone -q {REPO_URL}
%cd {REPO_NAME}

!pip install -q sentence-transformers python-dotenv

/content/DSA4264_Project


In [5]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Memory: {props.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


In [6]:
# 5. Run the embedding pipeline
#    --data-root bypasses .env (which doesn't exist on Colab)
#    --mode whole_text is the validated baseline
#    Reduce --batch-size to 32 or 16 if you hit GPU OOM

!python -m src.embedding.embed \
    --data-root "{DATA_ROOT}" \
    --mode whole_text \
    --batch-size 64

Loading modules from: /content/drive/MyDrive/DSA4264_Project_Data/processed/courses/modules_cleaned.csv
  → 8,615 modules
Loading jobs from: /content/drive/MyDrive/DSA4264_Project_Data/processed/jobs/03_jobs_filtered.csv
  → 13,663 jobs
Loading model: BAAI/bge-large-en-v1.5
Device: cuda (Tesla T4)
modules.json: 100% 349/349 [00:00<00:00, 1.69MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 486kB/s]
README.md: 94.6kB [00:00, 3.33MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 352kB/s]
config.json: 100% 779/779 [00:00<00:00, 4.49MB/s]
model.safetensors: 100% 1.34G/1.34G [00:07<00:00, 191MB/s]
Loading weights: 100% 391/391 [00:00<00:00, 1008.77it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

In [7]:
# 6. Verify outputs
import os

emb_dir = f"{DATA_ROOT}/embeddings/whole_text"
print(f"Output directory: {emb_dir}\n")

if os.path.exists(emb_dir):
    for fname in sorted(os.listdir(emb_dir)):
        size_mb = os.path.getsize(os.path.join(emb_dir, fname)) / (1024**2)
        print(f"  {fname:50s} {size_mb:>8.2f} MB")
else:
    print("  ERROR: Output directory not found.")

Output directory: /content/drive/MyDrive/DSA4264_Project_Data/embeddings/whole_text

  embedding_config.json                                  0.00 MB
  job_embeddings_bge-large-en-v1.5.npy                  53.37 MB
  job_index.csv                                          3.39 MB
  module_embeddings_bge-large-en-v1.5.npy               33.65 MB
  module_index.csv                                       0.74 MB


In [8]:
# 7. Quick sanity check
import numpy as np
import json

emb_dir = f"{DATA_ROOT}/embeddings/whole_text"

mod_emb = np.load(f"{emb_dir}/module_embeddings_bge-large-en-v1.5.npy")
job_emb = np.load(f"{emb_dir}/job_embeddings_bge-large-en-v1.5.npy")

with open(f"{emb_dir}/embedding_config.json") as f:
    config = json.load(f)

print(f"Module embeddings: {mod_emb.shape}")
print(f"Job embeddings:    {job_emb.shape}")
print(f"\nNorm check (should be ~1.0):")
print(f"  Modules: {np.linalg.norm(mod_emb, axis=1).mean():.4f}")
print(f"  Jobs:    {np.linalg.norm(job_emb, axis=1).mean():.4f}")
print(f"\nModel: {config['model']}")
print(f"Created: {config['created_at']}")

Module embeddings: (8615, 1024)
Job embeddings:    (13663, 1024)

Norm check (should be ~1.0):
  Modules: 1.0000
  Jobs:    1.0000

Model: BAAI/bge-large-en-v1.5
Created: 2026-04-10T18:43:37.446170


In [9]:
# 8. Download embeddings folder for local use / OneDrive sync
#    Uncomment and run when you're ready to download.

!zip -r /content/embeddings.zip "{DATA_ROOT}/embeddings/whole_text/"
from google.colab import files
files.download('/content/embeddings.zip')

  adding: content/drive/MyDrive/DSA4264_Project_Data/embeddings/whole_text/ (stored 0%)
  adding: content/drive/MyDrive/DSA4264_Project_Data/embeddings/whole_text/module_embeddings_bge-large-en-v1.5.npy (deflated 21%)
  adding: content/drive/MyDrive/DSA4264_Project_Data/embeddings/whole_text/job_embeddings_bge-large-en-v1.5.npy (deflated 10%)
  adding: content/drive/MyDrive/DSA4264_Project_Data/embeddings/whole_text/module_index.csv (deflated 83%)
  adding: content/drive/MyDrive/DSA4264_Project_Data/embeddings/whole_text/job_index.csv (deflated 79%)
  adding: content/drive/MyDrive/DSA4264_Project_Data/embeddings/whole_text/embedding_config.json (deflated 47%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>